# Jak właściwie muzykę charakteryzować?
Potrzebujemy zestawu cech które potem będziemy konkatenować i zamieniać na wektor danych
## Cechy barwowe - brzmienie
### MFCC (Mel-Frequency Cepstral Coefficients)

### Spectral Contrast

### Spectral Centroid

### Spectral Rolloff

### Zero Crossing Rate

## Cechy harmoniczne - tony, emocje
### Chroma Features

## Cechy rytmiczne 
### Tempo - bpm 

### Tempogram



# Ładowanie plików .mp3 
Funkcja ```librosa.load()``` Bierze ścieżke do pliku .mp3, podajemy mu sampling rate (22050 - to wartość w Hz, czyli na sekundę bierzemy 22050 punktów). Dla uproszczenia konwertujemy też sygnał na mono. Niestety, ale dla typowej piosenki wczytywanie trwa prawie minutę, więc spróbujemy innego rozwiązania

In [3]:
import librosa
import numpy as np
import os
import glob
import pickle
import subprocess
import numpy as np
import time

In [4]:
y, sr = librosa.load("data/Clipse-The_Birds_Don't_Sing.mp3", sr=22050, mono=True)

Robimy to samo - 22050Hz oraz dźwięk mono, ale używając ffmpeg, który jest napisany i zoptymalizowany w czystym C - to bardzo przyspiesza proces wczytywania. Przy okazji opakuje to w funkcję, która wszystkie pliki .mp3 z folderu data zapisze i wyeksportuje do pliku na którym potem będziemy pracować

In [8]:
def load_audio(file_path, sr=22050):
    command = [
        'ffmpeg',
        '-i', file_path,        # Plik wejściowy
        '-f', 'f32le',          # Format wyjściowy: float 32-bit 
        '-ac', '1',             # Audio Channels: 1 (Mono) 
        '-ar', str(sr),         # Audio Rate: docelowe próbkowanie
        '-acodec', 'pcm_f32le', # Kodek PCM
        '-'                     # Wyjście na standardowe wyjście (pipe) zamiast do pliku
    ]

    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, bufsize=10**8)
    stdout_data, _ = process.communicate()

    # Zamieniamy surowe bajty na tablicę numpy
    audio_array = np.frombuffer(stdout_data, dtype=np.float32)
    
    return audio_array

#Test działania funkcji dla pojedynczego pliku 

y = load_audio("data/Clipse-The_Birds_Don't_Sing.mp3")
print(f"Wczytano {len(y)} próbek.")


def preprocess_dataset(input_folder, output_file, target_sr=22050):
    files = glob.glob(os.path.join(input_folder, "*.mp3"))
    dataset = {}

    for i, path in enumerate(files):
        filename = os.path.basename(path)
        try:
            audio_data = load_audio(path, sr=target_sr)
            dataset[filename] = audio_data            
        except Exception as e:
            print(f"Błąd przy pliku {filename}: {e}")

    with open(output_file, 'wb') as f:
        pickle.dump(dataset, f)

preprocess_dataset('data', 'dataset.pkl')

Wczytano 5304832 próbek.


# Słowem wstępu
Aby wyliczyć spectral centroid, contrast i rollof potrzebuje wartości magnitud w czasie - wylicza się to wykorzystując algorytm FFT

## Standardowy FFT
FFT to algorytm który mając surowy sygnał wyliczy nam z jakich częstotliwości składa się sygnał

![fft](images/fft.png)

Problem - jeśli wrzucimy do tego algorytmu calą piosenke(sygnał) FFT wypisze nam częstotliwości wystepujące w całym utworze - bez podziału na kolejność

## Rozwiązanie STFT
Intuicja - dzielimy sygnał na okna czasowe i dla kazdego takiego przedziału wyliczamy FFT otrzymując macierz częstotliwosc x czas

In [9]:
def stft(audio_data,sr=22050, n_fft=2048, hop_length=512):
    """
    Ręczna implementacja STFT.
    input:
    - nfft: liczba próbek w kadej ramce, wieksze nfft - lepsza rozdzielczosc czestotliwosci ale gorsza czasowa, mniejsze nfft - na odwrót
    -hop_length: liczba próbek o które przesuwamy okno między kolejnymi ramkami
    output:
    - magnitudes: Macierz amplitud (częstotliwość x czas)
    - freq: tablica zakresu czestotliwosci
    """
    #1. Przygotowanie okna (Hanning Window) - statystyka: redukuje listki boczne (wyciek widma)
    window = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(n_fft) / n_fft)
    
    #2. Podział na ramki
    n_samples = len(audio_data)
    n_frames = 1 + (n_samples - n_fft) // hop_length
    
    # rfft zwraca n_fft/2 + 1 prążków częstotliwości (bo sygnał jest rzeczywisty, druga połowa to lustro)
    n_bins = n_fft // 2 + 1
    magnitudes = np.zeros((n_bins, n_frames))
    
    for i in range(n_frames):
        start = i * hop_length
        end = start + n_fft
        segment = audio_data[start:end]
        spectrum = np.fft.rfft(segment * window)
        
        #interesuje nas amplituda- wartość bezwzględna liczby zespolonej
        magnitudes[:, i] = np.abs(spectrum)

    freq = np.fft.rfftfreq(n_fft, d=1/sr)
        
    return magnitudes, freq

# Cechy barwowe:

### Spectral Centroid

To jest miara jasności dźwięku. Matematycznie jest to średnia ważona częstotliwości, gdzie wagami są amplitudy (energie) tych częstotliwości w danej chwili.

![eq](<images/spectral_centroid_eq.png>)
![desc](<images/spectral_centroid_desc.png>)


In [10]:
def compute_spectral_centroid(magnitudes,freq_bins):

    numerator = np.sum(magnitudes * freq_bins.reshape(-1, 1), axis=0)
    denominator = np.sum(magnitudes, axis=0)
    
    #dodajemy eps zeby na pewno nie podzielic przez zero
    eps = np.finfo(float).eps
    spectral_centroid = numerator / (denominator + eps)

    return np.array(spectral_centroid)

### Spectral Rolloff

Jest to częstotliwość Rt, poniżej której znajduje się określony procent (zazwyczaj 85%) całkowitej energii widma. Można to traktować jako kwantyl rzędu 0.85 rozkładu energii.

![eq](images/spectral_rolloff_eq.png)


In [11]:
def copmpute_spectral_rollof(magnitudes,freq_bins):
    threshold_percent = 0.85
    total_energy = np.sum(magnitudes, axis=0)
    threshold_energy = total_energy * threshold_percent
    
    #kumulujemy energię wzdłuż częstotliwości
    cumulative_energy = np.cumsum(magnitudes, axis=0)
    
    #szukamy indeksu, gdzie suma przekracza próg
    rolloff_indices = np.argmax(cumulative_energy >= threshold_energy, axis=0)
    spectral_rolloff = freq_bins[rolloff_indices]
    return np.array(spectral_rolloff)

### Spectral Contrast
    
Ta cecha dzieli widmo na pod-pasma (oktawy) i dla każdego pasma liczy różnicę między szczytami (peaks) a dolinami (valleys) energii.

In [12]:
def compute_spectral_contrast(magnitudes):
    #dziele widmo na 6 pasm i wyliczam dla kazdego pasma spectral contrast

    n_bands = 6
    n_bins = magnitudes.shape[0]
    band_size = n_bins // n_bands
    
    contrasts = []
    
    for i in range(n_bands):
        start = i * band_size
        end = (i + 1) * band_size
        band_magnitude = magnitudes[start:end, :]
        
        # Sortujemy amplitudy w paśmie, żeby znaleźć piki i doliny
        # Quantile method: alpha pika i alpha doliny
        peak = np.percentile(band_magnitude, 98, axis=0) # Górne 2%
        valley = np.percentile(band_magnitude, 2, axis=0) # Dolne 2%
        
        # Kontrast to różnica w skali logarytmicznej (dB)
        # Logarytmujemy, bo ludzkie ucho słyszy głośność logarytmicznie
        contrast = np.log1p(peak) - np.log1p(valley)
        contrasts.append(np.mean(contrast)) #średnia kontrastu w tym paśmie dla całego utworu

    return np.array(contrasts)



# Test

In [25]:
with open('dataset.pkl','rb') as f:
    dataset = pickle.load(f)

first_file_key = list(dataset.keys())[0]

magnitudes, freq = stft(dataset[first_file_key])

spectral_centroid = compute_spectral_centroid(magnitudes,freq)
spectral_rolloff = copmpute_spectral_rollof(magnitudes,freq)
spectral_contrast = compute_spectral_contrast(magnitudes)


print(spectral_centroid,spectral_centroid.shape)
print(spectral_rolloff,spectral_rolloff.shape)
print(spectral_contrast,spectral_contrast.shape)

[947.09066764 915.71470635 964.50292642 ...   0.           0.
   0.        ] (10358,)
[1636.5234375  1571.92382812 1604.22363281 ...    0.            0.
    0.        ] (10358,)
[3.45988111 1.76426232 1.11004561 0.951908   0.9161976  0.74743681] (6,)


# Problem do rozwiazania!

Kazda z tych cech jest wektorem liczb zaleznym od dlugosci wektora wejściowego (surowego sygnalu mp3) - z tego powodu w sumie to sa to trajektorie bardziej niz wektory ale whatever

Pierwsza propozycja naprawcza (dosyć prymitywna ale moze nie glupia) - dla kazdej takiej trajektorii liczymy srednia i wariancje i thats it

# --------------------------------------

# Cechy Harmoniczne

### Chroma features
mówi nam jak bardzo któryś z półtonów (C, C#, D, D#, E, F, F#, G, G#, A, A#, B) jest obecny w danym momencie utworu. Ignorujemy oktawy

Zwracamy macierz 12xT -> dla każdego okienka czasowego mamy wektor, mówiący jak bardzo któryś z półtonów jest obecny

Aby zamienić częstotliwość (Hz) na numer MIDI, używamy wzoru:

$$
m = 69 + 12 \cdot \log_2\left(\frac{f}{440}\right)
$$

gdzie:
- $f$ — częstotliwość w Hz,
- 69 — numer MIDI dla A4 (440 Hz),
- $\log_2$ — logarytm o podstawie 2 (liczba oktaw względem A4),
- mnożenie przez 12 zamienia oktawy na półtony.

In [ ]:
def compute_chroma(magnitudes, freqs):
    n_bins, n_frames = magnitudes.shape

    # Liczymy czestotliwosci MIDI dla wszystkich czestotliwosci na raz
    m = 69 + 12 * np.log2(freqs[1:] / 440.0)
    chroma_bins = np.round(m).astype(int) % 12  # 12 poltonow
    chroma_matrix = np.zeros((12, n_frames))

    for t in range(n_frames):
        np.add.at(chroma_matrix[:, t], chroma_bins, magnitudes[1:, t])

    chroma_matrix /= np.sum(chroma_matrix, axis=0, keepdims=True) + 1e-9

    return chroma_matrix

# --------------------------------------

# Cechy Rytmiczne

### Tempo - BPM
Liczba uderzeń na minutę, obliczamy tempo na podstawie STFT

$
\text{BPM} = \frac{60}{\text{czas między uderzeniami [s]}}
$

In [53]:
def estimate_bpm(magnitudes, sr=22050, hop_length=512, min_bpm=60, max_bpm=200):
    # ile energii mamy w danym okresie czasowym?
    energy = np.sum(magnitudes, axis=0)
    energy = (energy - np.mean(energy)) / (np.std(energy) + 1e-9) # normalizacja

    # porównujemy sygnał ze sobą po jakimś przesunięciu, sprawdzamy po jakim przeusnięciu
    # korelacja jest największa
    autocorr = np.correlate(energy, energy, mode='full')
    autocorr = autocorr[autocorr.size//2:]  # bierzemy tylko dodatnie lag

    # ograniczamy bpm do sensownego przedziału
    min_lag = int(sr * 60 / max_bpm / hop_length)
    max_lag = int(sr * 60 / min_bpm / hop_length)
    peak_index = np.argmax(autocorr[min_lag:max_lag]) + min_lag

    # zamieniamy na bpm
    period_seconds = peak_index * hop_length / sr
    bpm = 60 / period_seconds
    return bpm

In [54]:
chroma = compute_chroma(magnitudes, freq)
bpm = estimate_bpm(magnitudes)
print(chroma, chroma.shape)
print(f"BMP Utworu: {bpm}")

[0.00000000e+00 1.07666016e+01 2.15332031e+01 ... 1.10034668e+04
 1.10142334e+04 1.10250000e+04]
[[0.08811088 0.11311137 0.16573456 ... 0.         0.         0.        ]
 [0.05842164 0.04297366 0.04519653 ... 0.         0.         0.        ]
 [0.09457696 0.05834978 0.06049996 ... 0.         0.         0.        ]
 ...
 [0.1074309  0.06592695 0.07397389 ... 0.         0.         0.        ]
 [0.07410802 0.08207489 0.06574852 ... 0.         0.         0.        ]
 [0.10216408 0.1839258  0.16202149 ... 0.         0.         0.        ]] (12, 10358)
BMP Utworu: 86.1328125
